# Embedding Strategy Training & Evaluation

**Variants:** TF-IDF · Word2Vec · FastText · Doc2Vec · DistilBERT-Frozen · DistilBERT-FT · DistilBERT-MLM+FT  
**Seeds:** 42, 123, 456  
**Primary metric:** PR-AUC (extreme class imbalance ~0.95% train prevalence)

In [1]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(name)s  %(message)s',
    datefmt='%H:%M:%S',
)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Project root: {PROJECT_ROOT}')

Device: cpu
Project root: E:\Study\semester4\mlops\project\fraud-detection-mlops


## 1. Load Data

In [2]:
from src.training.train import load_data

train_df, val_df, test_df = load_data()

for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    n = len(df)
    n_fraud = df['is_fraud_receiver'].sum()
    print(f'{name:6s}: {n:>7,} accounts  {n_fraud:>5,} fraud  ({100*n_fraud/n:.2f}%)')

14:44:50  INFO      src.training.train  Loading tokenised splits from E:\Study\semester4\mlops\project\fraud-detection-mlops\data\processed\preprocessing
14:44:51  INFO      src.training.train  Loaded: train=356695  val=76434  test=76436


train : 356,695 accounts  3,384 fraud  (0.95%)
val   :  76,434 accounts  2,022 fraud  (2.65%)
test  :  76,436 accounts  2,763 fraud  (3.61%)


In [3]:
train_df.head()

,token_string,is_fraud_receiver,segment,seq_len
account_id,,,,
C1000004082,[CLS] [ACCT] SEG_REGULAR [TXN] CASH_OUT AMT_ME...,0,SEG_REGULAR,28
C1000004940,[CLS] [ACCT] SEG_HEAVY [TXN] CASH_OUT AMT_MEDI...,0,SEG_HEAVY,84
C100001587,[CLS] [ACCT] SEG_REGULAR [TXN] CASH_OUT AMT_VE...,0,SEG_REGULAR,44
C1000015936,[CLS] [ACCT] SEG_HEAVY [TXN] CASH_OUT AMT_MEDI...,0,SEG_HEAVY,68
C1000026379,[CLS] [ACCT] SEG_REGULAR [TXN] CASH_OUT AMT_LO...,0,SEG_REGULAR,28


## 2. Smoke Test — Single Variant


In [4]:
from src.training.train import run_experiment
from src.utils.mlflow_utils import setup_mlflow
from src.utils.config import CFG

mlflow_cfg = CFG.get('mlflow', {})
setup_mlflow(
    tracking_uri=mlflow_cfg.get('tracking_uri', 'file:./mlruns'),
    experiment_name=mlflow_cfg.get('experiment_name', 'fraud-detection-embeddings'),
)

# smoke_metrics = run_experiment('tfidf', seed=42, train_df=train_df, val_df=val_df,
#                                 test_df=test_df, device=device)
# print('\nSmoke test metrics:')
# for k, v in smoke_metrics.items():
#     if isinstance(v, float):
#         print(f'  {k}: {v:.4f}')

14:45:03  INFO      src.utils.mlflow_utils  MLflow configured: tracking_uri=http://localhost:5002  experiment=fraud-detection-embeddings (id=885488660584971937)
14:45:03  INFO      src.training.train  ============================================================
14:45:03  INFO      src.training.train  START  tfidf_seed42
14:45:03  INFO      src.training.train  ============================================================
14:45:07  INFO      src.models.tfidf_model  Fitting TF-IDF (max_features=5000, ngrams=(1, 3)) on 356695 sequences
14:45:27  INFO      src.models.tfidf_model  TF-IDF vocabulary size: 348
14:45:58  INFO      src.models.tfidf_model  TFIDFEmbedder saved → E:\Study\semester4\mlops\project\fraud-detection-mlops\model_artifacts\tfidf\seed_42\embedder.pkl
14:45:58  INFO      src.training.train  Embedding dim: 348
14:45:58  INFO      src.models.mlp_classifier  pos_weight=104.41 (n_pos=3384, n_neg=353311)
14:46:56  INFO      src.models.mlp_classifier  Epoch   1/50  loss=0.5502  va

🏃 View run tfidf_seed42 at: http://localhost:5002/#/experiments/885488660584971937/runs/400f1c08f9b84999aa2d5e25a2e0e3ab
🧪 View experiment at: http://localhost:5002/#/experiments/885488660584971937


KeyboardInterrupt: 

## 3. Full Training Run — All 7 Variants × 3 Seeds

In [5]:
from src.training.train import main, ALL_VARIANTS, ALL_SEEDS

# To run all 18 experiments:
# main(variants=ALL_VARIANTS, seeds=ALL_SEEDS)

# To run only statistical variants:
main(variants=['tfidf'], seeds=ALL_SEEDS)

14:48:15  INFO      src.training.train  Device: cpu
14:48:15  INFO      src.utils.mlflow_utils  MLflow configured: tracking_uri=http://localhost:5002  experiment=fraud-detection-embeddings (id=885488660584971937)
14:48:15  INFO      src.training.train  Loading tokenised splits from E:\Study\semester4\mlops\project\fraud-detection-mlops\data\processed\preprocessing
14:48:16  INFO      src.training.train  Loaded: train=356695  val=76434  test=76436
14:48:16  INFO      src.training.train  Subsampled train: 356695 → 54144  (fraud=3384 kept, non-fraud=50760/353311, ratio=1:15)
14:48:16  INFO      src.training.train  ============================================================
14:48:16  INFO      src.training.train  START  tfidf_seed42
14:48:16  INFO      src.training.train  ============================================================
14:48:18  INFO      src.models.tfidf_model  Fitting TF-IDF (max_features=5000, ngrams=(1, 3)) on 54144 sequences
14:48:21  INFO      src.models.tfidf_model  TF

🏃 View run tfidf_seed42 at: http://localhost:5002/#/experiments/885488660584971937/runs/c49e72ebaaaa4353b145648f62f5b367
🧪 View experiment at: http://localhost:5002/#/experiments/885488660584971937


14:50:46  INFO      src.models.tfidf_model  Fitting TF-IDF (max_features=5000, ngrams=(1, 3)) on 54144 sequences
14:50:49  INFO      src.models.tfidf_model  TF-IDF vocabulary size: 342
14:51:00  INFO      src.models.tfidf_model  TFIDFEmbedder saved → E:\Study\semester4\mlops\project\fraud-detection-mlops\model_artifacts\tfidf\seed_123\embedder.pkl
14:51:00  INFO      src.training.train  Embedding dim: 342
14:51:00  INFO      src.models.mlp_classifier  pos_weight=15.00 (n_pos=3384, n_neg=50760)
14:51:04  INFO      src.models.mlp_classifier  Epoch   1/50  loss=0.5062  val_pr_auc=0.7707
14:51:10  INFO      src.models.mlp_classifier  Epoch   2/50  loss=0.4200  val_pr_auc=0.7863
14:51:16  INFO      src.models.mlp_classifier  Epoch   3/50  loss=0.4085  val_pr_auc=0.7771
14:51:22  INFO      src.models.mlp_classifier  Epoch   4/50  loss=0.3973  val_pr_auc=0.7849
14:51:27  INFO      src.models.mlp_classifier  Epoch   5/50  loss=0.3937  val_pr_auc=0.7890
14:51:33  INFO      src.models.mlp_classi

🏃 View run tfidf_seed123 at: http://localhost:5002/#/experiments/885488660584971937/runs/0c7df4ae78044eb5bb86d9940531b7d5
🧪 View experiment at: http://localhost:5002/#/experiments/885488660584971937


14:52:16  INFO      src.models.tfidf_model  Fitting TF-IDF (max_features=5000, ngrams=(1, 3)) on 54144 sequences
14:52:19  INFO      src.models.tfidf_model  TF-IDF vocabulary size: 342
14:52:30  INFO      src.models.tfidf_model  TFIDFEmbedder saved → E:\Study\semester4\mlops\project\fraud-detection-mlops\model_artifacts\tfidf\seed_456\embedder.pkl
14:52:30  INFO      src.training.train  Embedding dim: 342
14:52:30  INFO      src.models.mlp_classifier  pos_weight=15.00 (n_pos=3384, n_neg=50760)
14:52:34  INFO      src.models.mlp_classifier  Epoch   1/50  loss=0.5027  val_pr_auc=0.7740
14:52:40  INFO      src.models.mlp_classifier  Epoch   2/50  loss=0.4194  val_pr_auc=0.7805
14:52:46  INFO      src.models.mlp_classifier  Epoch   3/50  loss=0.4069  val_pr_auc=0.7884
14:52:52  INFO      src.models.mlp_classifier  Epoch   4/50  loss=0.3986  val_pr_auc=0.7860
14:52:58  INFO      src.models.mlp_classifier  Epoch   5/50  loss=0.3971  val_pr_auc=0.7876
14:53:03  INFO      src.models.mlp_classi

🏃 View run tfidf_seed456 at: http://localhost:5002/#/experiments/885488660584971937/runs/01dfbe33c4544e5cafaaea4e0c4ab211
🧪 View experiment at: http://localhost:5002/#/experiments/885488660584971937


In [6]:
main(variants=['word2vec', 'fasttext', 'doc2vec'], seeds=ALL_SEEDS)

14:53:15  INFO      src.training.train  Device: cpu
14:53:15  INFO      src.utils.mlflow_utils  MLflow configured: tracking_uri=http://localhost:5002  experiment=fraud-detection-embeddings (id=885488660584971937)
14:53:15  INFO      src.training.train  Loading tokenised splits from E:\Study\semester4\mlops\project\fraud-detection-mlops\data\processed\preprocessing
14:53:16  INFO      src.training.train  Loaded: train=356695  val=76434  test=76436
14:53:16  INFO      src.training.train  Subsampled train: 356695 → 54144  (fraud=3384 kept, non-fraud=50760/353311, ratio=1:15)
14:53:16  INFO      src.training.train  ============================================================
14:53:16  INFO      src.training.train  START  word2vec_seed42
14:53:16  INFO      src.training.train  ============================================================
14:53:18  INFO      src.models.word2vec_model  Training Word2Vec (sg=1, dim=128, window=5, epochs=20) on 54144 sequences
14:53:18  INFO      gensim.models.w

🏃 View run word2vec_seed42 at: http://localhost:5002/#/experiments/885488660584971937/runs/600eee592a464ff997dc8e8ac1a45114
🧪 View experiment at: http://localhost:5002/#/experiments/885488660584971937


14:56:26  INFO      src.models.word2vec_model  Training Word2Vec (sg=1, dim=128, window=5, epochs=20) on 54144 sequences
14:56:26  INFO      gensim.models.word2vec  collecting all words and their counts
14:56:26  INFO      gensim.models.word2vec  PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
14:56:26  INFO      gensim.models.word2vec  PROGRESS: at sentence #10000, processed 461904 words, keeping 30 word types
14:56:27  INFO      gensim.models.word2vec  PROGRESS: at sentence #20000, processed 918584 words, keeping 30 word types
14:56:27  INFO      gensim.models.word2vec  PROGRESS: at sentence #30000, processed 1375968 words, keeping 30 word types
14:56:27  INFO      gensim.models.word2vec  PROGRESS: at sentence #40000, processed 1832480 words, keeping 30 word types
14:56:27  INFO      gensim.models.word2vec  PROGRESS: at sentence #50000, processed 2296792 words, keeping 30 word types
14:56:27  INFO      gensim.models.word2vec  collected 30 word types from a corpus of

🏃 View run word2vec_seed123 at: http://localhost:5002/#/experiments/885488660584971937/runs/972fe9a2e0f14fd29826f3199594c054
🧪 View experiment at: http://localhost:5002/#/experiments/885488660584971937


14:58:37  INFO      src.models.word2vec_model  Training Word2Vec (sg=1, dim=128, window=5, epochs=20) on 54144 sequences
14:58:37  INFO      gensim.models.word2vec  collecting all words and their counts
14:58:37  INFO      gensim.models.word2vec  PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
14:58:37  INFO      gensim.models.word2vec  PROGRESS: at sentence #10000, processed 461904 words, keeping 30 word types
14:58:37  INFO      gensim.models.word2vec  PROGRESS: at sentence #20000, processed 918584 words, keeping 30 word types
14:58:37  INFO      gensim.models.word2vec  PROGRESS: at sentence #30000, processed 1375968 words, keeping 30 word types
14:58:37  INFO      gensim.models.word2vec  PROGRESS: at sentence #40000, processed 1832480 words, keeping 30 word types
14:58:37  INFO      gensim.models.word2vec  PROGRESS: at sentence #50000, processed 2296792 words, keeping 30 word types
14:58:37  INFO      gensim.models.word2vec  collected 30 word types from a corpus of

...

🏃 View run doc2vec_seed42 at: http://localhost:5002/#/experiments/885488660584971937/runs/45c8252736024318ab29f02491adf043
🧪 View experiment at: http://localhost:5002/#/experiments/885488660584971937


15:13:26  INFO      src.models.doc2vec_model  Training Doc2Vec PV-DBOW (dim=128, window=5, epochs=20) on 54144 sequences
15:13:27  INFO      gensim.models.doc2vec  collecting all words and their counts
15:13:27  INFO      gensim.models.doc2vec  PROGRESS: at example #0, processed 0 words (0 words/s), 0 word types, 0 tags
15:13:27  INFO      gensim.models.doc2vec  PROGRESS: at example #10000, processed 461904 words (10961203 words/s), 30 word types, 0 tags
15:13:27  INFO      gensim.models.doc2vec  PROGRESS: at example #20000, processed 918584 words (10373998 words/s), 30 word types, 0 tags
15:13:27  INFO      gensim.models.doc2vec  PROGRESS: at example #30000, processed 1375968 words (10239222 words/s), 30 word types, 0 tags
15:13:27  INFO      gensim.models.doc2vec  PROGRESS: at example #40000, processed 1832480 words (10370346 words/s), 30 word types, 0 tags
15:13:27  INFO      gensim.models.doc2vec  PROGRESS: at example #50000, processed 2296792 words (10269821 words/s), 30 word type

🏃 View run doc2vec_seed123 at: http://localhost:5002/#/experiments/885488660584971937/runs/2721237488c94e5fb06d13f596135081
🧪 View experiment at: http://localhost:5002/#/experiments/885488660584971937


15:15:59  INFO      src.models.doc2vec_model  Training Doc2Vec PV-DBOW (dim=128, window=5, epochs=20) on 54144 sequences
15:15:59  INFO      gensim.models.doc2vec  collecting all words and their counts
15:15:59  INFO      gensim.models.doc2vec  PROGRESS: at example #0, processed 0 words (0 words/s), 0 word types, 0 tags
15:15:59  INFO      gensim.models.doc2vec  PROGRESS: at example #10000, processed 461904 words (11013867 words/s), 30 word types, 0 tags
15:15:59  INFO      gensim.models.doc2vec  PROGRESS: at example #20000, processed 918584 words (11300743 words/s), 30 word types, 0 tags
15:16:00  INFO      gensim.models.doc2vec  PROGRESS: at example #30000, processed 1375968 words (10739965 words/s), 30 word types, 0 tags
15:16:00  INFO      gensim.models.doc2vec  PROGRESS: at example #40000, processed 1832480 words (10357570 words/s), 30 word types, 0 tags
15:16:00  INFO      gensim.models.doc2vec  PROGRESS: at example #50000, processed 2296792 words (10837624 words/s), 30 word type

🏃 View run doc2vec_seed456 at: http://localhost:5002/#/experiments/885488660584971937/runs/355487fd490447d7bc62c0a55c9de29a
🧪 View experiment at: http://localhost:5002/#/experiments/885488660584971937


In [7]:
from src.training.train import main, ALL_VARIANTS, ALL_SEEDS

In [8]:
# frozen variant
main(variants=['distilbert_frozen'], seeds=[42, 123, 456])

15:17:37  INFO      src.training.train  Device: cpu
15:17:37  INFO      src.utils.mlflow_utils  MLflow configured: tracking_uri=http://localhost:5002  experiment=fraud-detection-embeddings (id=885488660584971937)
15:17:37  INFO      src.training.train  Loading tokenised splits from E:\Study\semester4\mlops\project\fraud-detection-mlops\data\processed\preprocessing
15:17:38  INFO      src.training.train  Loaded: train=356695  val=76434  test=76436
15:17:38  INFO      src.training.train  Subsampled train: 356695 → 54144  (fraud=3384 kept, non-fraud=50760/353311, ratio=1:15)
15:17:38  INFO      src.training.train  ============================================================
15:17:38  INFO      src.training.train  START  distilbert_frozen_seed42
15:17:38  INFO      src.training.train  ============================================================
15:17:40  INFO      src.models.distilbert_model  Loading DistilBERT tokeniser + model: distilbert-base-uncased
15:17:44  INFO      src.training.tra

🏃 View run distilbert_frozen_seed42 at: http://localhost:5002/#/experiments/885488660584971937/runs/fcfa079deb3048f1a64fc3366ef86fb9
🧪 View experiment at: http://localhost:5002/#/experiments/885488660584971937


18:19:24  INFO      src.models.distilbert_model  Loading DistilBERT tokeniser + model: distilbert-base-uncased
18:19:27  INFO      src.training.train  Loading cached BERT CLS vectors: train (54144 samples)
18:19:27  INFO      src.training.train  Loading cached BERT CLS vectors: val (76434 samples)
18:19:27  INFO      src.training.train  Loading cached BERT CLS vectors: test (76436 samples)
18:19:27  INFO      src.training.train  Applying seed-123 projection (Linear 768→128)…
18:19:27  INFO      src.models.distilbert_model  DistilBERTEmbedder saved → E:\Study\semester4\mlops\project\fraud-detection-mlops\model_artifacts\distilbert_frozen\seed_123\embedder
18:19:27  INFO      src.training.train  Embedding dim: 128
18:19:27  INFO      src.models.mlp_classifier  pos_weight=15.00 (n_pos=3384, n_neg=50760)
18:19:32  INFO      src.models.mlp_classifier  Epoch   1/50  loss=1.1356  val_pr_auc=0.4362
18:19:36  INFO      src.models.mlp_classifier  Epoch   2/50  loss=0.9684  val_pr_auc=0.4593
18:1

🏃 View run distilbert_frozen_seed123 at: http://localhost:5002/#/experiments/885488660584971937/runs/34e813e324264dec824e3fa2164312f8
🧪 View experiment at: http://localhost:5002/#/experiments/885488660584971937


18:23:31  INFO      src.models.distilbert_model  Loading DistilBERT tokeniser + model: distilbert-base-uncased
18:23:34  INFO      src.training.train  Loading cached BERT CLS vectors: train (54144 samples)
18:23:34  INFO      src.training.train  Loading cached BERT CLS vectors: val (76434 samples)
18:23:34  INFO      src.training.train  Loading cached BERT CLS vectors: test (76436 samples)
18:23:34  INFO      src.training.train  Applying seed-456 projection (Linear 768→128)…
18:23:35  INFO      src.models.distilbert_model  DistilBERTEmbedder saved → E:\Study\semester4\mlops\project\fraud-detection-mlops\model_artifacts\distilbert_frozen\seed_456\embedder
18:23:35  INFO      src.training.train  Embedding dim: 128
18:23:35  INFO      src.models.mlp_classifier  pos_weight=15.00 (n_pos=3384, n_neg=50760)
18:23:38  INFO      src.models.mlp_classifier  Epoch   1/50  loss=1.0582  val_pr_auc=0.4536
18:23:43  INFO      src.models.mlp_classifier  Epoch   2/50  loss=0.9306  val_pr_auc=0.4710
18:2

🏃 View run distilbert_frozen_seed456 at: http://localhost:5002/#/experiments/885488660584971937/runs/48ea7bd351324813b3600ec3ba504a67
🧪 View experiment at: http://localhost:5002/#/experiments/885488660584971937


In [ ]:
from src.training.train import main, ALL_VARIANTS, ALL_SEEDS
# finetune
main(variants=['distilbert_ft'], seeds=[42, 123, 456])

18:26:42  INFO      src.training.train  Device: cpu
18:26:42  INFO      src.utils.mlflow_utils  MLflow configured: tracking_uri=http://localhost:5002  experiment=fraud-detection-embeddings (id=885488660584971937)
18:26:42  INFO      src.training.train  Loading tokenised splits from E:\Study\semester4\mlops\project\fraud-detection-mlops\data\processed\preprocessing
18:26:43  INFO      src.training.train  Loaded: train=356695  val=76434  test=76436
18:26:43  INFO      src.training.train  Subsampled train: 356695 → 54144  (fraud=3384 kept, non-fraud=50760/353311, ratio=1:15)
18:26:43  INFO      src.training.train  ============================================================
18:26:43  INFO      src.training.train  START  distilbert_ft_seed42
18:26:43  INFO      src.training.train  ============================================================
18:26:44  INFO      src.training.train  === Fine-tuning distilbert_ft (seed=42) ===
18:26:46  INFO      src.models.distilbert_model  DistilBERT pos_wei

In [ ]:
# pretrain+finetune
main(variants=['distilbert_mlm_ft'], seeds=[42, 123, 456])

## 4. Load Results from MLflow

In [ ]:
import mlflow

experiment_name = CFG.get('mlflow', {}).get('experiment_name', 'fraud-detection-embeddings')
exp = mlflow.get_experiment_by_name(experiment_name)
runs_df = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="tags.phase = '4'",
)

print(f'Total runs: {len(runs_df)}')
metric_cols = [c for c in runs_df.columns if c.startswith('metrics.')]
tag_cols = ['tags.variant', 'tags.seed']
runs_df[tag_cols + metric_cols].sort_values(['tags.variant', 'tags.seed']).head(20)

In [ ]:
# Aggregate: mean ± std PR-AUC across 3 seeds per variant
VARIANT_ORDER = ['tfidf', 'word2vec', 'fasttext', 'doc2vec',
                 'distilbert_frozen', 'distilbert_ft', 'distilbert_mlm_ft']

summary = (
    runs_df
    .assign(variant=runs_df['tags.variant'])
    .groupby('variant')
    .agg(
        val_pr_auc_mean=('metrics.val_pr_auc', 'mean'),
        val_pr_auc_std=('metrics.val_pr_auc', 'std'),
        test_pr_auc_mean=('metrics.test_pr_auc', 'mean'),
        test_pr_auc_std=('metrics.test_pr_auc', 'std'),
        test_roc_auc_mean=('metrics.test_roc_auc', 'mean'),
        test_f1_mean=('metrics.test_f1', 'mean'),
        n_runs=('metrics.test_pr_auc', 'count'),
    )
    .reindex(VARIANT_ORDER)
    .reset_index()
)
summary

## 5. PR-AUC Comparison Plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(summary))
bars = ax.bar(
    x,
    summary['test_pr_auc_mean'],
    yerr=summary['test_pr_auc_std'],
    capsize=5,
    color=['#4878cf', '#4878cf', '#4878cf', '#4878cf', '#6acc65', '#d65f5f', '#b47cc7'],
    alpha=0.85,
    edgecolor='white',
)

# Annotate bar tops
for bar, mean in zip(bars, summary['test_pr_auc_mean']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.003,
        f'{mean:.3f}',
        ha='center', va='bottom', fontsize=9,
    )

ax.set_xticks(x)
ax.set_xticklabels(
    ['TF-IDF', 'Word2Vec', 'FastText', 'Doc2Vec',
     'DistilBERT\nFrozen', 'DistilBERT\nFT', 'DistilBERT\nMLM+FT'],
    fontsize=10,
)
ax.set_ylabel('Test PR-AUC (mean ± std, 3 seeds)', fontsize=11)
ax.set_title('Embedding Complexity vs. Fraud Detection Performance', fontsize=13, fontweight='bold')
ax.set_ylim(0, ax.get_ylim()[1] * 1.1)
ax.axhline(y=summary['test_pr_auc_mean'].iloc[0], ls='--', lw=0.8, color='grey', label='TF-IDF baseline')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
fig_path = PROJECT_ROOT / 'data' / 'processed' / 'fig_14_pr_auc_comparison.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved → {fig_path}')
plt.show()

## 6. Training Curves

In [ ]:
# Load training history from saved artifacts and plot val PR-AUC per epoch
# History is embedded in the MLflow run artifacts as JSON or available via
# loading the returned history dict from run_experiment().

# Example: re-run a single variant to capture history
from src.training.train import _run_statistical
from src.models.mlp_classifier import train_mlp

# To inspect training curves, re-run one variant with history capture:
# (history is the second return value from train_mlp)
print('Training curve inspection: re-run a specific variant above with verbose logging.')
print('The train_mlp() function logs per-epoch val_pr_auc to the console.')

## 7. Precision-Recall Curves Overlay

In [ ]:
from sklearn.metrics import precision_recall_curve
from src.models.mlp_classifier import load_mlp
from src.models.tfidf_model import TFIDFEmbedder
from src.models.word2vec_model import Word2VecEmbedder
from src.models.fasttext_model import FastTextEmbedder
from src.models.doc2vec_model import Doc2VecEmbedder

ARTIFACTS_PATH = PROJECT_ROOT / 'model_artifacts'
SEED = 42  # best-seed representative

test_strings = test_df['token_string'].tolist()
test_seqs = [s.split() for s in test_strings]
y_test = test_df['is_fraud_receiver'].values

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#4878cf', '#6acc65', '#d65f5f', '#f39c12', '#9b59b6', '#e74c3c', '#1abc9c']
labels = ['TF-IDF', 'Word2Vec', 'FastText', 'Doc2Vec',
          'DistilBERT-Frozen', 'DistilBERT-FT', 'DistilBERT-MLM+FT']

for variant, label, color in zip(
    ['tfidf', 'word2vec', 'fasttext', 'doc2vec',
     'distilbert_frozen', 'distilbert_ft', 'distilbert_mlm_ft'],
    labels, colors
):
    adir = ARTIFACTS_PATH / variant / f'seed_{SEED}'
    mlp_path = adir / 'mlp.pt'

    try:
        mlp = load_mlp(mlp_path, device)
        mlp.eval()

        if variant == 'tfidf':
            embedder = TFIDFEmbedder.load(adir / 'embedder.pkl')
            X_test = embedder.transform(test_strings)
        elif variant == 'word2vec':
            embedder = Word2VecEmbedder.load(adir / 'embedder.model')
            X_test = embedder.transform(test_seqs)
        elif variant == 'fasttext':
            embedder = FastTextEmbedder.load(adir / 'embedder.model')
            X_test = embedder.transform(test_seqs)
        elif variant == 'doc2vec':
            embedder = Doc2VecEmbedder.load(adir / 'embedder.model')
            X_test = embedder.transform(test_seqs)
        else:
            emb_dir = adir / 'embedder'
            if not emb_dir.exists():
                print(f'Skipping {variant}: embedder not found')
                continue
            from src.models.distilbert_model import DistilBERTEmbedder
            embedder = DistilBERTEmbedder.load(emb_dir, device)
            X_test = embedder.precompute_embeddings(test_strings)

        X_t = torch.tensor(X_test, dtype=torch.float32).to(device)
        with torch.no_grad():
            scores = torch.sigmoid(mlp(X_t)).cpu().numpy()

        prec, rec, _ = precision_recall_curve(y_test, scores)
        pr_auc = np.trapz(prec[::-1], rec[::-1])
        ax.plot(rec, prec, label=f'{label} (AUC={pr_auc:.3f})', color=color, lw=1.8)

    except FileNotFoundError:
        print(f'  {variant}: artifacts not found — run training first')

# Baseline: random classifier
prevalence = y_test.mean()
ax.axhline(prevalence, ls='--', color='grey', lw=0.8, label=f'Random ({prevalence:.3f})')

ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Precision-Recall Curves — Test Set (seed=42)', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
fig_path = PROJECT_ROOT / 'data' / 'processed' / 'fig_15_pr_curves.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved → {fig_path}')
plt.show()

## 8. Results Summary Table

In [ ]:
from src.training.evaluate import format_results_table

# Re-format summary for display
display_df = summary[[
    'variant',
    'val_pr_auc_mean', 'val_pr_auc_std',
    'test_pr_auc_mean', 'test_pr_auc_std',
    'test_roc_auc_mean', 'test_f1_mean',
    'n_runs',
]].copy()

display_df['val_PR-AUC'] = display_df.apply(
    lambda r: f"{r['val_pr_auc_mean']:.4f} ± {r['val_pr_auc_std']:.4f}", axis=1
)
display_df['test_PR-AUC'] = display_df.apply(
    lambda r: f"{r['test_pr_auc_mean']:.4f} ± {r['test_pr_auc_std']:.4f}", axis=1
)
display_df['test_ROC-AUC'] = display_df['test_roc_auc_mean'].map('{:.4f}'.format)
display_df['test_F1'] = display_df['test_f1_mean'].map('{:.4f}'.format)

final = display_df[['variant', 'val_PR-AUC', 'test_PR-AUC', 'test_ROC-AUC', 'test_F1', 'n_runs']]
print(final.to_string(index=False))

# Save to CSV
results_path = PROJECT_ROOT / 'data' / 'processed' / 'phase4_results.csv'
final.to_csv(results_path, index=False)
print(f'\nSaved → {results_path}')

## 9. Key Findings

Fill this in after running the full experiment:

- **Best variant:** `?` with test PR-AUC `?.????`  
- **TF-IDF baseline:** test PR-AUC `?.????`  
- **Doc2Vec vs Word2Vec:** tests whether document-level vectors outperform mean-pooled token vectors  
- **Improvement from DistilBERT-MLM+FT over TF-IDF:** `+?.????`  
- **Consistent with Phase 2–3 analysis:**
  - TIME_LATE_NIGHT (5.26×) and BAL_EXACT_DEBIT (2.20×) are the strongest fraud signals
  - DistilBERT variants should capture co-occurrence patterns TF-IDF misses
  - Class imbalance managed via `pos_weight ≈ 104.5` in BCEWithLogitsLoss